In [3]:
from ase.io import read, write

atoms = read('coded_JT_smooth_10.xyz', index=':')
print(len(atoms))

In [4]:
import numpy as np
from ase.neighborlist import NeighborList

def find_BX_vectors(atom, B_indices, X_indices, nl):
    nl.update(atom)
    BX_vectors = []
    for B_index in B_indices:
        index, offsets = nl.get_neighbors(B_index)
        BX_bonds = []
        for X_index, offset in zip(index, offsets):
            if X_index in X_indices:
                vec = atom.positions[X_index] + np.dot(offset, atom.get_cell()) - atom.positions[B_index]
                BX_bonds.append([vec, X_index])
        BX_vectors.append(BX_bonds)
    return BX_vectors

In [5]:
def angle_between_vectors(u, v):
    
    dot_product = np.dot(u, v)
    
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    
    cos_theta = dot_product / (norm_u * norm_v)
    
    angle_radians = np.arccos(cos_theta)
    angle_degrees = np.degrees(angle_radians)
    
    return angle_degrees

In [9]:
def find_long_bonds(atom, B_indices, X_indices, nl):
    
    BX_vectors = find_BX_vectors(atom, B_indices, X_indices, nl)
    JT_array = np.zeros(len(atom), dtype=int)
    long_array = np.zeros(len(atom), dtype=int)

    for B_index, BX_bonds in zip(B_indices, BX_vectors):
        sorted_bonds = sorted(BX_bonds, key=lambda x: np.linalg.norm(x[0]))
        long_bonds = [sorted_bonds[-1], sorted_bonds[-2]] # Get the two longest bonds
        angle = angle_between_vectors(long_bonds[0][0], long_bonds[1][0]) # Calculate the angle between the two longest bonds
        if 46 < angle < 135 or 230 < angle < 310:
            JT_array[B_index] = 4
            long_array[B_index] = B_index
            long_array[long_bonds[0][1]] = B_index
            long_array[long_bonds[1][1]] = B_index
        elif -45 < angle < 45 or 135 < angle < 220:
            x_angle = angle_between_vectors([1,0,0], long_bonds[0][0])
            y_angle = angle_between_vectors([0,1,0], long_bonds[0][0])
            z_angle = angle_between_vectors([0,0,1], long_bonds[0][0])
            if -40 < x_angle < 46 or 135 < x_angle < 220:
                JT_array[B_index] = 1
            elif -40 < y_angle < 46 or 135 < y_angle < 220:
                JT_array[B_index] = 2
            elif -40 < z_angle < 46 or 135 < z_angle < 220:
                JT_array[B_index] = 3
            else:
                print('B_index:', B_index, 'axis:', x_angle, y_angle, z_angle, 'lengths:', np.linalg.norm(long_bonds[0][0]), np.linalg.norm(long_bonds[1][0]),'long_bonds:', long_bonds, 'angle:', angle_between_vectors(long_bonds[0][0], long_bonds[1][0]))

            long_array[B_index] = B_index
            long_array[long_bonds[0][1]] = B_index
            long_array[long_bonds[1][1]] = B_index
        else:
            print('B_index:', B_index, 'lengths:', np.linalg.norm(long_bonds[0][0]), np.linalg.norm(long_bonds[1][0]),'long_bonds:', long_bonds, 'angle:', angle_between_vectors(long_bonds[0][0], long_bonds[1][0]))

    atom.arrays['long'] = long_array
    atom.arrays['JT'] = JT_array

    return atom

In [13]:
import time

B_site = 'Mn'
X_site = 'O'

B_indices = [site.index for site in atoms[0] if site.symbol == B_site]
X_indices = [site.index for site in atoms[0] if site.symbol == X_site]

cutoff = 3.0
cutoffs = [cutoff/2] * len(atoms[0])
nl = NeighborList(cutoffs, self_interaction=False, bothways=True, skin=0.0)

coded_atoms = []

windows = range(0, 100000, 10000)

for i, window in enumerate(windows):
    start = time.time()
    print("Structures:", windows[i-1], "to", window)
    for atom in atoms[windows[i-1]:window]:
        coded_atoms.append(find_long_bonds(atom, B_indices, X_indices, nl))
    end = time.time()
    print("Total time taken:", end - start)

In [16]:
write('coded_atoms_smooth_10.xyz', coded_atoms, format='extxyz')

In [10]:
number_of_x = []
number_of_z = []
number_of_l = []

for atom in coded_atoms:
    array = atom.arrays["JT"]
    
    xy = [1, 2]
    number_of_x.append(np.count_nonzero(np.isin(array, xy)))

    z = 3
    number_of_z.append(np.count_nonzero(array == z))

    l = 4
    number_of_l.append(np.count_nonzero(array == l))


In [16]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('/mnt/c/projects/LaMnO3/LaMnO3/Paper/paper.mplstyle')

plt.figure(figsize=(7.5, 1.3), dpi=300)

percentage_of_l = np.array(number_of_l) * np.array([100/216])
percentage_of_x = np.array(number_of_x) * np.array([100/216])
percentage_of_z = np.array(number_of_z) * np.array([100/216])

steps = np.linspace(200,1200,len(coded_atoms))

percentage_roll_x =   pd.Series(percentage_of_x).rolling(500, min_periods=0, win_type="hamming").mean()
percentage_roll_z =   pd.Series(percentage_of_z).rolling(500, min_periods=0, win_type="hamming").mean()
percentage_roll_l =   pd.Series(percentage_of_l).rolling(500, min_periods=0, win_type="hamming").mean()

plt.plot(steps, percentage_of_l, color="#EE3377", alpha=0.2)
plt.plot(steps, percentage_of_x, color="#117733", alpha=0.2)
plt.plot(steps, percentage_of_z, color="#33BBEE", alpha=0.2)

plt.plot(steps, percentage_roll_l,label='L-type', color="#EE3377")
plt.plot(steps, percentage_roll_x,label='X-axis', color="#117733")
plt.plot(steps, percentage_roll_z,label='Z-axis', color="#33BBEE")

plt.gca().xaxis.set_label_position('bottom')

plt.ylim(-10, 110)
plt.xlim(300, 1100)
plt.xticks([400, 600, 800, 1000])
plt.xlabel('Temperature (K)')
plt.ylabel('Percentage of octahedra (%)')

plt.savefig("two_long_bonds_evo_smooth_10_v3.png", bbox_inches='tight')
plt.show()